# ReAct-M: action batches
This notebook runs the same package as the CLI. Install with `python -m pip install -e .` first. All executed cells use hand-authored plans and fictional documents; no model API is called.


In [1]:
import json

from react_m.experiments import compare, run_case

## A data dependency
The search result supplies the document ID directly to the read action. Inspect the trace to see both calls inside one batch.


In [2]:
demo = await run_case("dependency", "multi", tool_delay=0, decision_delay=0)
print(json.dumps(demo, indent=2))

{
  "status": "finished",
  "answer": "1892",
  "mode": "multi",
  "backend": "scripted",
  "decision_calls": 2,
  "model_calls": 0,
  "tool_attempts": 2,
  "input_tokens": null,
  "output_tokens": null,
  "usage_complete": true,
  "history": [
    {
      "decision": {
        "type": "execute",
        "actions": [
          {
            "id": "search",
            "tool": "search",
            "args": {
              "query": "Alder Observatory"
            }
          },
          {
            "id": "read",
            "tool": "read",
            "bindings": {
              "doc_id": {
                "action": "search",
                "pointer": "/matches",
                "select": "only",
                "project": "/id"
              }
            }
          }
        ]
      },
      "observation": {
        "status": "complete",
        "results": [
          {
            "id": "search",
            "status": "succeeded",
            "attempts": 1,
            "elapsed_s

## Three modes and negative controls
Zero configured delays make this a correctness demonstration, not a latency benchmark. Use the CLI comparison with declared delays for mechanics measurements.


In [3]:
comparison = await compare(repeats=1, tool_delay=0, decision_delay=0)
for row in comparison["summary"]:
    print(
        row["scenario"],
        row["mode"],
        "success:",
        row["success_rate"],
        "decisions:",
        row["mean_decisions"],
        "real model calls:",
        row["mean_model_calls"],
    )

independent single success: 1.0 decisions: 3 real model calls: 0
independent parallel success: 1.0 decisions: 2 real model calls: 0
independent multi success: 1.0 decisions: 2 real model calls: 0
dependency single success: 1.0 decisions: 3 real model calls: 0
dependency parallel success: 1.0 decisions: 3 real model calls: 0
dependency multi success: 1.0 decisions: 2 real model calls: 0
ambiguous single success: 1.0 decisions: 2 real model calls: 0
ambiguous parallel success: 1.0 decisions: 2 real model calls: 0
ambiguous multi success: 1.0 decisions: 2 real model calls: 0
failure single success: 1.0 decisions: 2 real model calls: 0
failure parallel success: 1.0 decisions: 2 real model calls: 0
failure multi success: 1.0 decisions: 2 real model calls: 0


## Real-model evaluation
Configure `OPENAI_API_KEY`, `OPENAI_MODEL` and optionally `OPENAI_BASE_URL`, then run `python -m react_m evaluate --output results/local/live.json` to collect model responses, tool calls and outcomes for the same fixtures. See the [README](README.md#compare-execution-modes) for the execution modes and configuration.
